# QuantJourney SDK - News Filings Price Reaction Panel

This notebook demonstrates a QuantJourney SDK workflow that combines financial news, SEC filings, earnings dates, insider transactions and adjusted prices into an issuer event-reaction panel.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbol = 'AAPL'
peer_symbols = ['AAPL', 'MSFT', 'NVDA', 'GOOGL']
tiingo_news_raw = qj.tiingo.get_news(tickers=symbol, startDate='2024-01-01', endDate=END)
finnhub_news_raw = qj.finnhub.get_company_news(symbol=symbol, from_date='2024-01-01', to_date=END)
filings_raw = qj.sec.get_company_filings(symbol=symbol, limit=30)
insiders_raw = qj.sec.get_insider_transactions(symbol=symbol, limit=100)
earnings_raw = qj.fmp.get_earnings_calendar(from_date='2024-01-01', to_date=END)
prices, volumes = price_panel(peer_symbols, start='2023-01-01', end=END)


In [ ]:
events = []
for source, payload in {'Tiingo news': tiingo_news_raw, 'Finnhub company news': finnhub_news_raw, 'SEC filings': filings_raw, 'SEC insider transactions': insiders_raw, 'FMP earnings calendar': earnings_raw}.items():
    for row in as_rows(payload):
        raw_date = row.get('date') or row.get('datetime') or row.get('publishedDate') or row.get('filingDate') or row.get('transactionDate')
        if isinstance(raw_date, (int, float)):
            date = pd.to_datetime(raw_date, errors='coerce', unit='s')
        else:
            date = pd.to_datetime(raw_date, errors='coerce')
        if pd.notna(date):
            events.append({'source': source, 'event_date': date.normalize(), 'raw': row})
events = pd.DataFrame(events)
if events.empty:
    raise RuntimeError('No issuer event rows returned')


In [ ]:
px = prices[symbol].dropna()
reaction_rows = []
for row in events.itertuples():
    pos = px.index.searchsorted(row.event_date)
    if pos > 1 and pos + 5 < len(px):
        reaction_rows.append({'source': row.source, 'event_date': row.event_date, 'reaction_1d': px.iloc[pos + 1] / px.iloc[pos] - 1, 'reaction_5d': px.iloc[pos + 5] / px.iloc[pos] - 1})
reactions = pd.DataFrame(reaction_rows)
source_summary = reactions.groupby('source')[['reaction_1d', 'reaction_5d']].median().sort_values('reaction_5d')
display(pd.Series({'event_rows': len(events), 'reaction_rows': len(reactions), 'peer_price_columns': prices.shape[1]}))
display(source_summary)
source_summary.plot(kind='barh', title='Median AAPL price reaction by public event source')
plt.xlabel('return')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.